In [13]:
from datasets import load_dataset
import pandas as pd
import os
import requests
import logging


In [71]:
dataset = load_dataset("PatronusAI/financebench")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
        num_rows: 150
    })
})


In [76]:
type(dataset)

datasets.dataset_dict.DatasetDict

In [16]:
train_data


Dataset({
    features: ['financebench_id', 'company', 'doc_name', 'question_type', 'question_reasoning', 'domain_question_num', 'question', 'answer', 'justification', 'dataset_subset_label', 'evidence', 'gics_sector', 'doc_type', 'doc_period', 'doc_link'],
    num_rows: 150
})

In [79]:

# Initialize a dictionary to hold the data
data = {
    "financebench_id": [],
    "doc_name": [],
    "doc_link": [],
    "doc_period": [],
    "question_type": [],
    "question": [],
    "answer": [],
    "evidence": [],
    "doc_link" : []
}

# Extract data from the 'train' split
train_data = dataset['train']

# Loop through each row and append data to the respective lists
for row in train_data:
    data["financebench_id"].append(row["financebench_id"])
    data["doc_name"].append(row["doc_name"])
    data["doc_link"].append(row["doc_link"])
    data["doc_period"].append(row["doc_period"])
    data["question_type"].append(row["question_type"])
    data["question"].append(row["question"])
    data["answer"].append(row["answer"])
    data["evidence"].append(row["evidence"])
    data["doc_link"].append(row["doc_link"])
# Convert the data dictionary to a pandas DataFrame
input_data = pd.DataFrame(train_data)




In [80]:
input_data

,financebench_id,company,doc_name,question_type,question_reasoning,domain_question_num,question,answer,justification,dataset_subset_label,evidence,gics_sector,doc_type,doc_period,doc_link
0,financebench_id_03029,3M,3M_2018_10K,metrics-generated,Information extraction,None,What is the FY2018 capital expenditure amount ...,$1577.00,The metric capital expenditures was directly e...,OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
1,financebench_id_04672,3M,3M_2018_10K,metrics-generated,Information extraction,None,Assume that you are a public equities analyst....,$8.70,"The metric ppne, net was directly extracted fr...",OPEN_SOURCE,[{'evidence_text': 'Table of Contents 3M Comp...,Industrials,10k,2018,https://investors.3m.com/financials/sec-filing...
2,financebench_id_00499,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is 3M a capital-intensive business based on FY...,"No, the company is managing its CAPEX and Fixe...",CAPEX/Revenue\nFixed Assets/Total Assets\nROA=...,OPEN_SOURCE,[{'evidence_text': '3M Company and Subsidiarie...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
3,financebench_id_01226,3M,3M_2022_10K,domain-relevant,Logical reasoning (based on numerical reasonin...,dg17,What drove operating margin change as of FY202...,Operating Margin for 3M in FY2022 has decrease...,None,OPEN_SOURCE,"[{'evidence_text': 'SG&A, measured as a percen...",Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
4,financebench_id_01865,3M,3M_2022_10K,novel-generated,None,None,"If we exclude the impact of M&A, which segment...",The consumer segment shrunk by 0.9% organically.,None,OPEN_SOURCE,[{'evidence_text': 'Worldwide Sales Change By...,Industrials,10k,2022,https://investors.3m.com/financials/sec-filing...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,financebench_id_00215,Verizon,VERIZON_2022_10K,domain-relevant,Logical reasoning (based on numerical reasoning),dg06,Is Verizon a capital intensive business based ...,Yes. Verizon's capital intensity ratio was app...,capital intensity ratio = total asset / revenu...,OPEN_SOURCE,[{'evidence_text': 'Consolidated Balance Sheet...,Communication Services,10k,2022,https://www.verizon.com/about/sites/default/fi...
146,financebench_id_00566,Verizon,VERIZON_2022_10K,domain-relevant,Numerical reasoning,dg22,Has Verizon increased its debt on balance shee...,No. Verizon's debt decreased by $229 million.,debt change = debt in 2022 - debt in 2021 = 15...,OPEN_SOURCE,"[{'evidence_text': 'At December 31, Maturities...",Communication Services,10k,2022,https://www.verizon.com/about/sites/default/fi...
147,financebench_id_06247,Walmart,WALMART_2018_10K,metrics-generated,Numerical reasoning,None,What is FY2018 days payable outstanding (DPO) ...,42.69,The metric in question was calculated using ot...,OPEN_SOURCE,[{'evidence_text': 'Walmart Inc. Consolidated ...,Consumer Staples,10k,2018,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000...
148,financebench_id_04784,Walmart,WALMART_2019_10K,metrics-generated,Numerical reasoning,None,Based on the information provided primarily in...,0.2%,The metric in question was calculated using ot...,OPEN_SOURCE,[{'evidence_text': 'WalmartInc. ConsolidatedSt...,Consumer Staples,10k,2019,https://d18rn0p25nwr6d.cloudfront.net/CIK-0000...


In [8]:
# Directory where PDFs will be saved
output_dir = r'E:\RAG_Project\data\raw\pdf_reports'
os.makedirs(output_dir, exist_ok=True)
# Set up logging
logging.basicConfig(filename='download_log.txt', level=logging.ERROR, 
                    format='%(asctime)s:%(levelname)s:%(message)s')

# Function to download and save PDF
def download_pdf(url, save_path):
    try:
        response = requests.get(url)
        response.raise_for_status()  # Check if the request was successful
        with open(save_path, 'wb') as file:
            file.write(response.content)
        print(f"Downloaded: {save_path}")
    except requests.RequestException as e:
        logging.error(f"Failed to download {url}: {e}")
        print(f"Failed to download {url}. Check log for details.")

# Iterate over each row in the DataFrame and download the PDFs
for idx, row in input_data.iterrows():
    pdf_url = row['doc_link']
    pdf_filename = os.path.join(output_dir, f"{row['doc_name']}.pdf")
    if not os.path.exists(pdf_filename):
        download_pdf(pdf_url, pdf_filename)
    else:
        print(f"File already exists: {pdf_filename}")

print("All PDFs have been processed.")

File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2018_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2018_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2022_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2022_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2022_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2023Q2_10Q.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2023Q2_10Q.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\3M_2023Q2_10Q.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\ACTIVISIONBLIZZARD_2019_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\ACTIVISIONBLIZZARD_2019_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\ADOBE_2015_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\ADOBE_2016_10K.pdf
File already exists: E:\RAG_Project\data\raw\pdf_reports\ADOBE_2017_10K.pdf
File

In [9]:
git clone https://github.com/patronus-ai/financebench.git


SyntaxError: invalid syntax (744104230.py, line 1)

In [19]:
import requests

def get_10k_html_url(company_name: str, year: int, api_key: str):
    # Step 1: Search for the 10-K using SEC-API
    query_url = "https://api.sec-api.io/company-filings"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {
        "ticker": company_name,
        "formType": "10-K",
        "pageSize": 1,
        "startDate": f"{year}-01-01",
        "endDate": f"{year}-12-31"
    }

    response = requests.get(query_url, headers=headers, params=params)
    if response.status_code != 200:
        raise Exception(f"SEC API request failed: {response.status_code}")

    data = response.json()

    if not data["filings"]:
        print("No 10-K found for this year and company.")
        return None

    filing = data["filings"][0]
    accession_number = filing["accessionNumber"].replace("-", "")
    cik = filing["cik"]
    primary_doc = filing["primaryDocument"]

    # Step 2: Construct the HTML URL
    html_url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_number}/{primary_doc}"
    return html_url


In [31]:
import requests
import time
from bs4 import BeautifulSoup

# Headers requeridos por la SEC
HEADERS = {
    "User-Agent": "MyRAGApp/1.0 (rafaelgutierrez.n@gmail.com)",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

def get_cik(company_name):
    """
    Devuelve el CIK con padding de 10 dígitos para la empresa dada.
    """
    url = "https://www.sec.gov/files/company_tickers.json"
    response = requests.get(url, headers=HEADERS)
    companies = response.json()

    for v in companies.values():
        if company_name.lower() in v["title"].lower():
            cik_str = str(v["cik_str"]).zfill(10)
            return cik_str
    return None

def get_10k_accession_and_doc(cik, year):
    """
    Busca el número de acceso y el documento principal del 10-K para un año.
    """
    submissions_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(submissions_url, headers=HEADERS)
    data = response.json()

    for i, form_type in enumerate(data["filings"]["recent"]["form"]):
        if form_type == "10-K":
            filing_date = data["filings"]["recent"]["filingDate"][i]
            if str(year) in filing_date:
                accession = data["filings"]["recent"]["accessionNumber"][i].replace("-", "")
                primary_doc = data["filings"]["recent"]["primaryDocument"][i]
                return accession, primary_doc
    return None, None

def download_10k_html(company_name, year, save_path="10k_report.html"):
    cik = get_cik(company_name)
    if not cik:
        print(f"❌ No se encontró CIK para {company_name}")
        return

    accession_number, primary_doc = get_10k_accession_and_doc(cik, year)
    if not accession_number:
        print(f"❌ No se encontró el 10-K para {company_name} en {year}")
        return

    html_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_number}/{primary_doc}"
    print(f"✅ Descargando: {html_url}")

    response = requests.get(html_url, headers=HEADERS)
    if response.status_code == 200:
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(response.text)
        print(f"✅ Guardado en {save_path}")
    else:
        print(f"❌ Error al descargar el HTML: {response.status_code}")

# ✅ Ejemplo de uso
download_10k_html("3M", 2015, "3m_10k_2015.html")



JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [9]:

import json
import requests

def get_cik_for_company(company_name, mapping=None):
    """
    Return the 10-digit CIK string for a given company name.
    Mapping should be a dict of {company_name: cik_str}.
    """
    default_mapping = {
        "Adobe": "0000796343",
        "3M": "0000066740",
        # Add more companies and their CIKs here
    }
    mapping = mapping or default_mapping
    cik = mapping.get(company_name)
    if not cik:
        raise ValueError(f"CIK for company '{company_name}' not found in mapping.")
    return cik.zfill(10)


def fetch_filings_json(cik_padded):
    """
    Fetch the company submission JSON from the SEC EDGAR API.
    """
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    headers = {"User-Agent": "Your Name rafaelgutierrez.n@gmail.com"}
    resp = requests.get(url, headers=headers)
    resp.raise_for_status()
    return resp.json()


def generate_sec_urls(jsonl_path, company_cik_map=None):
    """
    Reads a JSONL file of document metadata, looks up the SEC accession
    for each 10-K filing by year, and constructs the HTML report URLs.

    Args:
        jsonl_path (str): Path to the JSONL file containing document info.
        company_cik_map (dict): Optional mapping of company names to CIKs.

    Returns:
        List[str]: A list of SEC report URLs.
    """
    urls = []

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        for line in f:
            doc = json.loads(line)
            company = doc.get('company')
            year = doc.get('doc_period')
            raw_cik = doc.get('cik')

            try:
                if raw_cik:
                    cik_padded = str(raw_cik).zfill(10)
                else:
                    cik_padded = get_cik_for_company(company, mapping=company_cik_map)
            except ValueError as e:
                print(f"Warning: {e}")
                continue

            recent = fetch_filings_json(cik_padded)["filings"]["recent"]
            forms = recent.get('form', [])
            accession_numbers = recent.get('accessionNumber', [])
            primary_docs = recent.get('primaryDocument', [])
            fiscal_years = recent.get('fiscalYear', [])

            # Find the 10-K for the specified fiscal year using safe iteration
            url = None
            for form, acc_no, doc_filename, fy in zip(forms, accession_numbers, primary_docs, fiscal_years):
                if form.upper() == '10-K' and fy == year:
                    acc = acc_no.replace('-', '')
                    cik_int = str(int(cik_padded))
                    url = f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{acc}/{doc_filename}"
                    break

            if url:
                urls.append(url)
            else:
                print(f"Warning: 10-K for {company} in {year} not found.")

    return urls


In [10]:


if __name__ == '__main__':
    path = r"E:\RAG_Project\data\financebench\data\financebench_document_information.jsonl"
    custom_mapping = {
        "Adobe": "0000796343",
        "3M": "0000066740",
        # ... add other companies as needed
    }
    urls = generate_sec_urls(path, company_cik_map=custom_mapping)
    for u in urls:
        print(u)

In [24]:
response

{'total': {'value': 20, 'relation': 'eq'},
 'query': {'from': 0, 'size': 200},
 'filings': [{'ticker': 'BCAL',
   'formType': 'DEF 14A',
   'accessionNo': '0001795815-25-000003',
   'cik': '1795815',
   'companyNameLong': 'California BanCorp \\ CA (Filer)',
   'companyName': 'California BanCorp \\ CA',
   'linkToFilingDetails': 'https://www.sec.gov/Archives/edgar/data/1795815/000179581525000003/bcal2025proxystatement.htm',
   'description': 'Form DEF 14A - Other definitive proxy statements',
   'linkToTxt': 'https://www.sec.gov/Archives/edgar/data/1795815/000179581525000003/0001795815-25-000003.txt',
   'filedAt': '2025-04-08T16:15:29-04:00',
   'documentFormatFiles': [{'sequence': '1',
     'size': '721998',
     'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1795815/000179581525000003/bcal2025proxystatement.htm',
     'description': 'DEF 14A',
     'type': 'DEF 14A'},
    {'sequence': '2',
     'size': '6479',
     'documentUrl': 'https://www.sec.gov/Archives/edgar/data/1795

In [70]:
import json
import requests
from pathlib import Path
from tqdm import tqdm
from difflib import get_close_matches

# Ruta al archivo .jsonl
jsonl_path = Path("E:/RAG_Project/data/financebench/data/financebench_document_information.jsonl")

# Descargar el archivo de CIKs
tickers_url = "https://www.sec.gov/files/company_tickers.json"
headers = {
    "User-Agent": "MyApp/1.0 (rafaelgutierrez.n@gmail.com)",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}
print("🔄 Descargando listado de compañías...")
resp = requests.get(tickers_url, headers=headers)
company_map = {
    entry["title"].strip().upper(): f'{entry["cik_str"]:010d}'
    for entry in resp.json().values()
}
print(f"✅ {len(company_map)} compañías cargadas.")

# Función de búsqueda flexible
def fuzzy_lookup(name, reference_dict):
    matches = get_close_matches(name, reference_dict.keys(), n=1, cutoff=0.7)
    if matches:
        return reference_dict[matches[0]], matches[0]
    return None, None

# Leer el archivo jsonl
print(f"📂 Procesando archivo: {jsonl_path}")
with open(jsonl_path, "r", encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]

# Filtrar documentos 10-K
ten_k_docs = [d for d in docs if d["doc_type"].lower() == "10k"]

# Buscar URLs
result = []
for doc in tqdm(ten_k_docs, desc="🔍 Buscando reportes 10-K"):
    company = doc["company"].strip().upper()
    year = int(doc["doc_period"])

    cik, matched_name = fuzzy_lookup(company, company_map)
    if not cik:
        print(f"❌ CIK no encontrado para {company}")
        continue
    else:
        if company != matched_name:
            print(f"🔗 Emparejado: {company} → {matched_name}")

    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    try:
        response = requests.get(url, headers=headers)
        submission_data = response.json()
        found = False

        for acc, form, date in zip(
            submission_data["filings"]["recent"]["accessionNumber"],
            submission_data["filings"]["recent"]["form"],
            submission_data["filings"]["recent"]["filingDate"],
        ):
            if form == "10-K" and date.startswith(str(year)):
                acc_no = acc.replace("-", "")
                html_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no}/{acc_no}-index.htm"
                print(f"✅ {company} {year} → {html_url}")
                result.append({
                    "company": company,
                    "matched": matched_name,
                    "year": year,
                    "cik": cik,
                    "html_url": html_url
                })
                found = True
                break

        if not found:
            print(f"⚠️ 10-K no encontrado para {company} ({cik}) en {year}")

    except Exception as e:
        print(f"❌ Error al procesar {company}: {e}")

# Guardar resultados
output_path = jsonl_path.parent / "financebench_10k_urls.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(result, f, indent=2)

print(f"\n✅ URLs guardadas en: {output_path}")



🔄 Descargando listado de compañías...
✅ 7629 compañías cargadas.
📂 Procesando archivo: E:\RAG_Project\data\financebench\data\financebench_document_information.jsonl


🔍 Buscando reportes 10-K:   4%|▎         | 10/269 [00:00<00:03, 65.81it/s]

❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para 3M
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD


🔍 Buscando reportes 10-K:   6%|▋         | 17/269 [00:00<00:06, 40.05it/s]

❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ACTIVISION BLIZZARD
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
❌ CIK no encontrado para ADOBE
🔗 Emparejado: AES CORPORATION → HILLS BANCORPORATION


...

🔍 Buscando reportes 10-K:  88%|████████▊ | 237/269 [00:16<00:02, 10.77it/s]

❌ Error al procesar PFIZER: Expecting value: line 1 column 1 (char 0)
🔗 Emparejado: PFIZER → PFIZER INC
❌ Error al procesar PFIZER: Expecting value: line 1 column 1 (char 0)
🔗 Emparejado: PFIZER → PFIZER INC
❌ Error al procesar PFIZER: Expecting value: line 1 column 1 (char 0)


🔍 Buscando reportes 10-K:  88%|████████▊ | 237/269 [00:16<00:02, 14.68it/s]


🔗 Emparejado: PFIZER → PFIZER INC


KeyboardInterrupt: 

In [67]:
# En lugar de requests.get(...)
with open("E:/RAG_Project/data/company_tickers.json", "r", encoding="utf-8") as f:
    company_data = json.load(f)

company_map = {
    entry["title"].strip().upper(): f'{entry["cik_str"]:010d}'
    for entry in company_data.values()
}
company_map

{'MICROSOFT CORP': '0000789019',
 'APPLE INC.': '0000320193',
 'NVIDIA CORP': '0001045810',
 'AMAZON COM INC': '0001018724',
 'ALPHABET INC.': '0001652044',
 'META PLATFORMS, INC.': '0001326801',
 'BERKSHIRE HATHAWAY INC': '0001067983',
 'TESLA, INC.': '0001318605',
 'BROADCOM INC.': '0001730168',
 'TAIWAN SEMICONDUCTOR MANUFACTURING CO LTD': '0001046179',
 'WALMART INC.': '0000104169',
 'ELI LILLY & CO': '0000059478',
 'JPMORGAN CHASE & CO': '0000019617',
 'VISA INC.': '0001403161',
 'UNITEDHEALTH GROUP INC': '0000731766',
 'SPDR S&P 500 ETF TRUST': '0000884394',
 'MASTERCARD INC': '0001141391',
 'EXXON MOBIL CORP': '0000034088',
 'COSTCO WHOLESALE CORP /NEW': '0000909832',
 'NETFLIX INC': '0001065280',
 'ORACLE CORP': '0001341439',
 'PROCTER & GAMBLE CO': '0000080424',
 'JOHNSON & JOHNSON': '0000200406',
 'HOME DEPOT, INC.': '0000354950',
 'ABBVIE INC.': '0001551152',
 'SAP SE': '0001000184',
 'COCA COLA CO': '0000021344',
 'NOVO NORDISK A S': '0000353278',
 'T-MOBILE US, INC.': '000

In [8]:
#!/usr/bin/env python3
"""
fetch_sec_10k_html.py

Lee un archivo JSONL de información de documentos,
para cada doc_type 10-K de 3M y Activision Blizzard:
- Intenta obtener la URL HTML usando la API de EDGAR (json submissions)
  (busca en "recent" con condición en el nombre del documento si no coincide la fecha)
- Si no encuentra, usa la página índice EDGAR para extraer el enlace 10-K HTML
Imprime ambas URL o informa error.
"""

import json
import time
import requests
from bs4 import BeautifulSoup

# Ruta al JSONL
JSONL_PATH = r"E:\RAG_Project\data\financebench\data\financebench_document_information.jsonl"
# Compañías a procesar con su CIK de 10 dígitos
COMPANY_INFO = {
    "3M": {"cik10": "0001558370"},
    "Activision Blizzard": {"cik10": "0000718877"},
}

# Headers estándar para EDGAR
HEADERS = {
    "User-Agent": "fetch_sec_10k_html.py (rafaelgutierrez.n@gmail.com)",
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "Accept-Language": "en-US,en;q=0.5",
}


def fetch_submissions(cik10):
    """Devuelve el JSON de submissions para un CIK"""
    url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    resp = requests.get(url, headers=HEADERS)
    resp.raise_for_status()
    return resp.json()


def find_10k_api(subs, year):
    """Busca en recent filings el 10-K de un año (o documento que incluya el año en su nombre)"""
    recent = subs.get("filings", {}).get("recent", {})
    forms        = recent.get("form", [])
    dates        = recent.get("filingDate", [])
    accessions   = recent.get("accessionNumber", [])
    primary_docs = recent.get("primaryDoc", [])
    for form, date, acc, doc in zip(forms, dates, accessions, primary_docs):
        if form == "10-K" and (date.startswith(str(year)) or str(year) in doc):
            return acc, doc
    return None, None


def extract_accession_from_pdf(pdf_url):
    """Toma el nombre del PDF y lo usa como accession"""
    filename = pdf_url.rstrip('/').split('/')[-1]
    return filename.replace('.pdf', '')


def build_index_url(accession):
    """Construye URL de página índice EDGAR"""
    cik = accession.split('-')[0].lstrip('0')
    nohyphen = accession.replace('-', '')
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{nohyphen}/{accession}-index.html"


def fetch_html_from_index(index_url):
    """Scrapea la página índice y extrae el enlace 10-K HTML"""
    resp = requests.get(index_url, headers=HEADERS)
    if resp.status_code != 200:
        return None
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', {'summary': 'Document Format Files'})
    if not table:
        return None
    for row in table.find_all('tr'):
        cols = row.find_all('td')
        if len(cols) >= 3 and cols[1].get_text(strip=True) == '10-K':
            href = cols[2].find('a')['href']
            return 'https://www.sec.gov' + href
    return None


def build_html_url_from_api(cik10, acc, doc):
    """Genera la URL HTML usando API discovery"""
    cik_int = str(int(cik10))
    nohyphen = acc.replace('-', '')
    return f"https://www.sec.gov/Archives/edgar/data/{cik_int}/{nohyphen}/{doc}"


def main():
    submissions_cache = {}

    with open(JSONL_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            rec = json.loads(line)
            company = rec.get('company')
            year = rec.get('doc_period')
            if company in COMPANY_INFO and rec.get('doc_type', '').lower() == '10k':
                print(f"Procesando {company} {year} 10-K:")
                info = COMPANY_INFO[company]
                cik10 = info['cik10']

                # 1) Intento API
                if cik10 not in submissions_cache:
                    try:
                        submissions_cache[cik10] = fetch_submissions(cik10)
                    except Exception as e:
                        submissions_cache[cik10] = None
                        print(f"  WARNING: falló fetch_submissions(): {e}")
                    time.sleep(0.2)

                subs = submissions_cache[cik10]
                html_url = None
                if subs:
                    acc, doc = find_10k_api(subs, year)
                    if acc:
                        html_url = build_html_url_from_api(cik10, acc, doc)
                        print(f"  API → {html_url}")

                # 2) Fallback: scrappear índice si no se obtuvo vía API
                if not html_url:
                    pdf_link = rec.get('doc_link')
                    accession = extract_accession_from_pdf(pdf_link)
                    idx_url = build_index_url(accession)
                    print(f"  Fallback index: {idx_url}")
                    html_url = fetch_html_from_index(idx_url)
                    if html_url:
                        print(f"  Índice → {html_url}")
                    else:
                        print("  ERROR: no se encontró 10-K HTML ni vía API ni scrapping.")

                print()
                time.sleep(0.2)

if __name__ == '__main__':
    main()




Procesando 3M 2015 10-K:
  Fallback index: https://www.sec.gov/Archives/edgar/data/1558370/000155837016003162/0001558370-16-003162-index.html
  Índice → https://www.sec.gov/Archives/edgar/data/66740/000155837016003162/mmm-20151231x10k.htm

Procesando 3M 2016 10-K:
  Fallback index: https://www.sec.gov/Archives/edgar/data/1558370/000155837017000479/0001558370-17-000479-index.html
  Índice → https://www.sec.gov/Archives/edgar/data/66740/000155837017000479/mmm-20161231x10k.htm

Procesando 3M 2017 10-K:
  Fallback index: https://www.sec.gov/Archives/edgar/data/1558370/000155837018000535/0001558370-18-000535-index.html
  Índice → https://www.sec.gov/Archives/edgar/data/66740/000155837018000535/mmm-20171231x10k.htm

Procesando 3M 2018 10-K:
  Fallback index: https://www.sec.gov/Archives/edgar/data/1558370/000155837019000470/0001558370-19-000470-index.html
  Índice → https://www.sec.gov/Archives/edgar/data/66740/000155837019000470/mmm-20181231x10k.htm

Procesando 3M 2019 10-K:
  Fallback inde